# Fine-tune Qwen3.5-4B three ways, measure them one way, serve the winner

<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/qwen3.5-fine-tuning/qwen3.5-fine-tuning-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

Full fine-tune, LoRA and QLoRA on the same support-ticket task, scored on the same held-out rows by the same function, with the cost of each (VRAM, minutes, bytes) on the same row as the lift. Then the adapter is served next to the base on the same vLLM endpoint.

| Step | What happens |
|---|---|
| 0 | `baseline`: the untouched model on 100 held-out tickets. Run this first; a task the base already does is not worth fine-tuning |
| 1 | `pipeline --method lora`: data, train with a live loss curve, evaluate base vs fine-tuned |
| 2 | The same with `qlora` and `full` |
| 3 | `compare`: all three in parallel on three GPUs, one report |
| 4 | `serve.py`: the adapter as its own model name on vLLM |

The task: a customer message in, `{"intent": ..., "reply": ...}` out, intent one of 27 labels the base model has never seen named. Two recomputable scores per row (intent exact match, format kept) and an optional pairwise, position-swapped judge for the reply.

---

## Setup

### Attach an accelerator first

- **Colab:** Runtime → Change runtime type → **T4 GPU** (or better) before running anything below. `qlora` fits a T4; `lora` and `full` need a 48 GB card, so run those on a cluster.
- **Apple Silicon Mac:** start the devbox with `flyte start devbox --apple-silicon` and put `GPU=apple` in `.env` (or `GPU=apple flyte run ...`). The GPU tasks then run on the Mac's own GPU, `lora` only.
- **NVIDIA machine:** `flyte start devbox --gpu`.
- **Union cluster:** nothing to attach; `GPU=L40s` is the default.

Without one of these the GPU tasks fall back to the CPU and a 4B model will not finish.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/qwen3.5-fine-tuning
    !uv pip install -r requirements.txt && (python -c 'import torch' 2>/dev/null || uv pip install -r requirements-local.txt)
    !uv pip install keyrings.alt
    !mkdir -p ~/.config/python_keyring && echo -e '[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring' > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

### Where the GPU work runs

On a cluster or devbox the GPU tasks run in containers (reports, run graph). With `--local` they run in this process on Colab's GPU: `qlora` fits a 16 GB T4; `lora` and `full` need a 48 GB card.

In [ ]:
# Optional: connect to a Flyte or Union cluster, or a devbox on your machine.
# Skip this to use `--local` cells only.
#
# Don't have a cluster? Request demo access at https://union.ai/
#
# !flyte create config \
#     --endpoint <your-endpoint> \
#     --project flytesnacks \
#     --domain development \
#     --builder remote

In [ ]:
# Optional judge for the replies (pairwise, blind, position-swapped). Needs an Anthropic key.
import os
# os.environ["JUDGE"] = "claude-haiku-5-5"
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."     # --local; on a cluster, create the ANTHROPIC_API_KEY secret instead

### Running the workflows

Every step is a plain `flyte run`, so the usual flags work:

| | |
|---|---|
| `flyte run --local ...` | Runs in this Python process. No cluster, no image build, no reports. |
| `flyte run ...` | Runs on the cluster or devbox you configured above: containers, the run graph and the live reports in the UI. |

Open the run URL that `flyte run` prints to see the report: that is where the charts are.

### The knobs

In [ ]:
!cat config.py

### 0. Baseline first

The per-category bars in the report show where the gap is.

In [ ]:
!flyte run workflow.py baseline
# no cluster: !flyte run --local workflow.py baseline --num_eval_examples 20

### 1. LoRA

Open the `train` task for the live loss curve and the discovered LoRA targets; `evaluate` for base vs fine-tuned.

In [ ]:
!flyte run workflow.py pipeline --method lora
# no cluster (16 GB GPU): !flyte run --local workflow.py pipeline --method qlora --train_samples 200 --num_eval_examples 40

### 2. QLoRA and full fine-tuning

Same data, same scorer. What changes is what you pay.

In [ ]:
!flyte run workflow.py pipeline --method qlora

In [ ]:
!flyte run workflow.py pipeline --method full

### 3. All three, one report

In [ ]:
!flyte run workflow.py compare

### 4. Serve the adapter

The fine-tuned model becomes `qwen3.5-4b-support` next to the base on the same OpenAI-compatible endpoint.

In [ ]:
!python serve.py                 # latest pipeline run
# !python serve.py --run <run name> --method full

## What to take away

- Run the baseline before you train. The lift is on the labels that need the dataset's definitions; the per-category chart shows where.
- Three methods, one measurement. The table in `compare` is lift next to VRAM, minutes and bytes.
- Recomputable scores first; the judge last, pairwise and position-swapped so its bias is visible as ties.
- LoRA targets are discovered from the module tree. Qwen3.5 is three Gated DeltaNet blocks per attention block; a copied target list would skip most of it.